# KramaBench Experiment Analysis - Complete vs. Without SUTs
This notebook performs a comprehensive comparative analysis of the `OpenCode_hierarchical_index_complete` and `OpenCode_hierarchical_index_without` SUTs across the `archeology`, `biomedical`, and `legal` workloads.

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('.'))
import re
import json
import glob
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from benchmark.metrics import Success

# Set pandas options for nice markdown output
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)

# Exclude specific SUT directories
excluded_suts = [
    'OpenCode_hierarchical_index_complete_R0_old_prompt',
    'OpenCode_hierarchical_index_complete_partial_test_R1',
    'OpenCode_hierarchical_index_complete_ds_pro'
]

In [2]:
def get_canonical_files(workload):
    path = os.path.join('data', workload, 'input')
    files = []
    if os.path.exists(path):
        for root, dirs, filenames in os.walk(path):
            for f in filenames:
                rel = os.path.relpath(os.path.join(root, f), path)
                files.append(rel.replace('\\', '/'))
    return files

def matches_source(pred_file, gold_source):
    p = pred_file.replace('\\', '/').lower()
    g = gold_source.replace('\\', '/').lower()
    if g.endswith('/*'):
        prefix = g[:-2]
        return p.startswith(prefix)
    return p == g or os.path.basename(p) == os.path.basename(g)

def compute_precision_recall_f1(code_path, gold_sources, canon_files):
    if not code_path or not os.path.exists(code_path):
        return 0.0, 0.0, 0.0
    with open(code_path, 'r', encoding='utf-8', errors='ignore') as f:
        code = f.read().lower()
    
    pred_set = set()
    for canon_f in canon_files:
        basename = os.path.basename(canon_f)
        if canon_f.lower() in code or basename.lower() in code:
            pred_set.add(canon_f)
            
    tp = 0
    for pf in pred_set:
        if any(matches_source(pf, gs) for gs in gold_sources):
            tp += 1
    precision = tp / len(pred_set) if len(pred_set) > 0 else 1.0
    
    matched_gold = 0
    for gs in gold_sources:
        if any(matches_source(pf, gs) for pf in pred_set):
            matched_gold += 1
    recall = matched_gold / len(gold_sources) if len(gold_sources) > 0 else 1.0
    
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def parse_session_file(filepath):
    if not filepath or not os.path.exists(filepath):
        return {
            'exploration': 0,
            'analysis': 0,
            'overhead': 0,
            'skill_invocations': {'expand': 0, 'search': 0, 'summarize': 0}
        }
        
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    turns = content.split("## Assistant")[1:]
    
    exploration_chars = 0
    analysis_chars = 0
    overhead_chars = 0
    skill_invocations = {'expand': 0, 'search': 0, 'summarize': 0}
    
    for turn in turns:
        thinking_match = re.search(r'_Thinking:_(.*?)(?:\*\*Tool:|$)', turn, re.DOTALL)
        thinking_text = thinking_match.group(1).strip() if thinking_match else ""
        
        tool_match = re.search(r'\*\*Tool:\s*(\w+)\*\*', turn)
        tool_name = tool_match.group(1).strip().lower() if tool_match else None
        
        input_match = re.search(r'\*\*Input:\*\*\s*```(?:json|python)?\s*(.*?)\s*```', turn, re.DOTALL)
        input_text = input_match.group(1).strip() if input_match else ""
        
        phase = 'Exploration'
        if tool_name == 'bash':
            if 'timer.py' in input_text:
                phase = 'Overhead'
            elif 'python' in input_text or 'python.exe' in input_text:
                phase = 'Analysis'
            else:
                phase = 'Exploration'
        elif tool_name == 'write':
            phase = 'Analysis'
        elif tool_name in ['read', 'search', 'skill']:
            phase = 'Exploration'
        
        turn_len = len(thinking_text) + len(input_text)
        if phase == 'Exploration':
            exploration_chars += turn_len
        elif phase == 'Analysis':
            analysis_chars += turn_len
        else:
            overhead_chars += turn_len
            
        if input_text:
            for s in skill_invocations.keys():
                matches = re.findall(rf'\b{s}\b', input_text.lower())
                skill_invocations[s] += len(matches)
                
    return {
        'exploration': exploration_chars,
        'analysis': analysis_chars,
        'overhead': overhead_chars,
        'skill_invocations': skill_invocations
    }

def find_session_file(sut_dir, task_index, workload):
    prefix = 'a' if workload == 'archeology' else ('b' if workload == 'biomedical' else 'l')
    files = glob.glob(os.path.join(sut_dir, f"session-*-{prefix}-{task_index}.md"))
    if not files:
        files = glob.glob(os.path.join(sut_dir, f"session-*-{task_index}.md"))
    for f in files:
        with open(f, 'r', encoding='utf-8', errors='ignore') as file:
            content = file.read()
        if f'Workload name: {workload}' in content:
            return f
    return None

def get_token_count(sut_dir, task_index, workload):
    prefix = 'a' if workload == 'archeology' else ('b' if workload == 'biomedical' else 'l')
    token_csv = os.path.join(sut_dir, f"token-{prefix}.csv")
    if not os.path.exists(token_csv):
        return 0
    try:
        df = pd.read_csv(token_csv)
        row = df[(df['workload'] == workload) & (df['task'] == task_index)]
        if not row.empty:
            return int(row.iloc[0]['token'])
    except Exception:
        pass
    return 0

In [3]:
def parse_pof(filepath):
    if not filepath or not os.path.exists(filepath):
        return {}
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    tasks = content.split("### ")[1:]
    res = {}
    for t in tasks:
        lines = t.strip().split('\n')
        task_id = lines[0].strip()
        m = re.search(r'\*\*Failure Type:\*\*\s*`?([^`\n\r]+)`?', t)
        if m:
            res[task_id] = m.group(1).strip()
    return res

def get_pof_type(sut_dir, task_id, workload):
    candidates = [
        os.path.join(sut_dir, f"point_of_failure_{workload}.md"),
        os.path.join(sut_dir, "point_of_failure.md"),
        os.path.join(sut_dir, "point_of_failure_archeology.md"),
        os.path.join(sut_dir, "point_of_failure_biomedical.md"),
        os.path.join(sut_dir, "point_of_failure_legal.md")
    ]
    for c in candidates:
        if os.path.exists(c):
            pof_map = parse_pof(c)
            if task_id in pof_map:
                return pof_map[task_id]
    return None

In [4]:
# Load workloads
workloads = {}
for w in ['archeology', 'biomedical', 'legal']:
    with open(f'workload/{w}.json', 'r') as f:
        workloads[w] = json.load(f)

# SUT configurations mapping
configs = []
for r in ['R1', 'R2', 'R3']:
    configs.append(('archeology', f'complete_{r}', f'complete_{r}', f'OpenCode_hierarchical_index_complete_{r}'))
    configs.append(('archeology', f'without_{r}', f'without_{r}', f'OpenCode_hierarchical_index_without_{r}'))

configs.append(('biomedical', 'complete_R1', 'complete_R1', 'OpenCode_hierarchical_index_complete_R2'))
configs.append(('biomedical', 'without_R1', 'without_R1', 'OpenCode_hierarchical_index_without_R1'))

configs.append(('legal', 'complete_R1', 'complete_R1', 'OpenCode_hierarchical_index_complete_R2'))
configs.append(('legal', 'without_R1', 'without_R1', 'OpenCode_hierarchical_index_without'))

success_metric = Success()
records = []

for wl_name, sut_report_name, sut_label, sut_dir_name in configs:
    sut_dir = os.path.join("system_scratch", sut_dir_name)
    canon_files = get_canonical_files(wl_name)
    
    for idx, task in enumerate(workloads[wl_name]):
        task_index = idx + 1
        tid = task['id']
        target = task['answer']
        
        ans_path = os.path.join(sut_dir, tid, "answer.txt")
        success = 0
        if os.path.exists(ans_path):
            with open(ans_path, 'r', encoding='utf-8', errors='ignore') as f:
                pred = f.read().strip()
            try:
                success, _, _, _ = success_metric(pred, target)
            except Exception:
                success = 0
                
        tokens = get_token_count(sut_dir, task_index, wl_name)
        sess_path = find_session_file(sut_dir, task_index, wl_name)
        sess_metrics = parse_session_file(sess_path)
        
        code_path = os.path.join(sut_dir, tid, "pipeline_code.py")
        precision, recall, f1 = compute_precision_recall_f1(code_path, task['data_sources'], canon_files)
        
        pof_type = get_pof_type(sut_dir, tid, wl_name)
        
        records.append({
            'workload': wl_name,
            'sut_report': sut_report_name,
            'sut_label': sut_label,
            'task_id': tid,
            'success': success,
            'tokens': tokens,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'explore_chars': sess_metrics['exploration'],
            'analyze_chars': sess_metrics['analysis'],
            'overhead_chars': sess_metrics['overhead'],
            'expand_calls': sess_metrics['skill_invocations']['expand'],
            'search_calls': sess_metrics['skill_invocations']['search'],
            'summarize_calls': sess_metrics['skill_invocations']['summarize'],
            'pof_type': pof_type
        })

df_all = pd.DataFrame(records)
df_all.to_csv('repro_fle_2/experiment_analysis_results.csv', index=False)
print(f"Generated and saved experiment_analysis_results.csv for {len(df_all)} SUT run-tasks.")

Traceback (most recent call last):
  File "D:\Fadhil\Pelajaran\Riset\ACE\RESEARCH\KramaBench\KramaBench-R5\repro_fle_2\generate_ipynb.py", line 499, in <module>
    exec(code_str, g_dict, g_dict)
  File "<string>", line 70, in <module>
  File "C:\Users\Fadhilah\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\util\_decorators.py", line 333, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Fadhilah\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\generic.py", line 3986, in to_csv
    return DataFrameRenderer(formatter).to_csv(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Fadhilah\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\io\formats\format.py", line 1014, in to_csv
    csv_formatter.save()
  File "C:\Users\Fadhilah\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\io\formats\csvs.py", line 251, in save
    with get_handle(
         ^^^^^^^^^^^
  File

In [5]:
def get_mean_std_str(series):
    mean_val = series.mean() * 100
    if len(series) > 1:
        std_val = series.std() * 100
        return f"{mean_val:.2f}% +- {std_val:.2f}%"
    return f"{mean_val:.2f}%"

# Calculate run accuracies
run_accs = df_all.groupby(['workload', 'sut_report'])['success'].mean().reset_index()

print("### SUT Workload Performance (Mean +- Std Dev)\n")
summary_rows = []
for wl in ['archeology', 'biomedical', 'legal']:
    for sut_type in ['complete', 'without']:
        sub = run_accs[(run_accs['workload'] == wl) & (run_accs['sut_report'].str.startswith(sut_type))]
        summary_rows.append({
            'Workload': wl,
            'SUT Type': sut_type,
            'Accuracy': get_mean_std_str(sub['success']),
            'Runs Included': list(sub['sut_report'])
        })
df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_markdown(index=False))

print("\n### Global Workload-Weighted Average Accuracy\n")
global_rows = []
for sut_type in ['complete', 'without']:
    wl_means = []
    for wl in ['archeology', 'biomedical', 'legal']:
        sub = run_accs[(run_accs['workload'] == wl) & (run_accs['sut_report'].str.startswith(sut_type))]
        wl_means.append(sub['success'].mean())
    wl_means = np.array(wl_means)
    global_rows.append({
        'SUT Type': sut_type,
        'Weighted Mean Accuracy': f"{wl_means.mean()*100:.2f}% +- {wl_means.std()*100:.2f}%"
    })
df_global = pd.DataFrame(global_rows)
print(df_global.to_markdown(index=False))

print("\n### Task-by-Task Success Details\n")
task_pivot = df_all.pivot(index='task_id', columns='sut_report', values='success')
print(task_pivot.to_markdown())

### SUT Workload Performance (Mean +- Std Dev)

| Workload   | SUT Type   | Accuracy        | Runs Included                                 |
|:-----------|:-----------|:----------------|:----------------------------------------------|
| archeology | complete   | 41.67% +- 8.33% | ['complete_R1', 'complete_R2', 'complete_R3'] |
| archeology | without    | 36.11% +- 4.81% | ['without_R1', 'without_R2', 'without_R3']    |
| biomedical | complete   | 44.44%          | ['complete_R1']                               |
| biomedical | without    | 55.56%          | ['without_R1']                                |
| legal      | complete   | 53.33%          | ['complete_R1']                               |
| legal      | without    | 56.67%          | ['without_R1']                                |

### Global Workload-Weighted Average Accuracy

| SUT Type   | Weighted Mean Accuracy   |
|:-----------|:-------------------------|
| complete   | 46.48% +- 4.98%          |
| without    | 49.44% +- 9

In [6]:
# Summarize Precision, Recall, and F1 file usage
file_metrics = df_all.groupby(['workload', 'sut_report'])[['precision', 'recall', 'f1']].mean().reset_index()

file_summary = []
for wl in ['archeology', 'biomedical', 'legal']:
    for sut_type in ['complete', 'without']:
        sub = file_metrics[(file_metrics['workload'] == wl) & (file_metrics['sut_report'].str.startswith(sut_type))]
        file_summary.append({
            'Workload': wl,
            'SUT Type': sut_type,
            'Mean Precision': f"{sub['precision'].mean():.4f}",
            'Mean Recall': f"{sub['recall'].mean():.4f}",
            'Mean F1-score': f"{sub['f1'].mean():.4f}"
        })
df_file_summary = pd.DataFrame(file_summary)
print("### File Usage Metrics Comparison\n")
print(df_file_summary.to_markdown(index=False))

### File Usage Metrics Comparison

| Workload   | SUT Type   |   Mean Precision |   Mean Recall |   Mean F1-score |
|:-----------|:-----------|-----------------:|--------------:|----------------:|
| archeology | complete   |           1      |        0.9167 |          0.9444 |
| archeology | without    |           1      |        0.9167 |          0.9444 |
| biomedical | complete   |           0.9556 |        0.9444 |          0.9352 |
| biomedical | without    |           0.9722 |        0.9444 |          0.9471 |
| legal      | complete   |           0.7222 |        0.7    |          0.6911 |
| legal      | without    |           0.7167 |        0.6611 |          0.6522 |


In [7]:
# Compute Pearson correlation coefficients
corr_acc_f1, _ = pearsonr(df_all['success'], df_all['f1'])
corr_acc_tokens, _ = pearsonr(df_all['success'], df_all['tokens'])
corr_f1_tokens, _ = pearsonr(df_all['f1'], df_all['tokens'])

print("### Task-Level Pearson Correlation Coefficients\n")
print(f"- **Accuracy vs. F1 File Usage:** {corr_acc_f1:.4f}")
print(f"- **Accuracy vs. Token Cost:** {corr_acc_tokens:.4f}")
print(f"- **F1 File Usage vs. Token Cost:** {corr_f1_tokens:.4f}")

### Task-Level Pearson Correlation Coefficients

- **Accuracy vs. F1 File Usage:** 0.2809
- **Accuracy vs. Token Cost:** -0.3632
- **F1 File Usage vs. Token Cost:** -0.2153


In [8]:
# Skill invocation stats
skill_cols = ['expand_calls', 'search_calls', 'summarize_calls']
df_complete = df_all[df_all['sut_report'].str.startswith('complete')]

skill_summary = []
for col in skill_cols:
    skill_name = col.replace('_calls', '')
    total_calls = df_complete[col].sum()
    mean_calls = df_complete[col].mean()
    tasks_using = (df_complete[col] > 0).mean() * 100
    skill_summary.append({
        'Skill': skill_name,
        'Total Calls': total_calls,
        'Mean Calls/Task': f"{mean_calls:.2f}",
        'Tasks Utilizing Skill (%)': f"{tasks_using:.2f}%"
    })
df_skills = pd.DataFrame(skill_summary)
print("### Skill Invocation Statistics (Complete SUT Runs Only)\n")
print(df_skills.to_markdown(index=False))

print("\n### Skill Invocations per Task ID\n")
print(df_complete[['task_id', 'sut_report', 'expand_calls', 'search_calls', 'summarize_calls']].to_markdown(index=False))

### Skill Invocation Statistics (Complete SUT Runs Only)

| Skill     |   Total Calls |   Mean Calls/Task | Tasks Utilizing Skill (%)   |
|:----------|--------------:|------------------:|:----------------------------|
| expand    |            72 |              0.96 | 33.33%                      |
| search    |           112 |              1.49 | 26.67%                      |
| summarize |            10 |              0.13 | 5.33%                       |

### Skill Invocations per Task ID

| task_id            | sut_report   |   expand_calls |   search_calls |   summarize_calls |
|:-------------------|:-------------|---------------:|---------------:|------------------:|
| archeology-hard-1  | complete_R1  |              5 |              7 |                 0 |
| archeology-hard-2  | complete_R1  |              0 |              1 |                 0 |
| archeology-easy-3  | complete_R1  |              0 |              0 |                 0 |
| archeology-easy-4  | complete_R1  |         

In [9]:
# Point of Failure analysis
df_failed = df_all[df_all['success'] == 0]

print("### Task-by-Task Failure Classifications\n")
df_pof_task = df_failed[['task_id', 'sut_report', 'pof_type']].drop_duplicates()
df_pof_task = df_pof_task.sort_values(by=['task_id', 'sut_report'])
print(df_pof_task.to_markdown(index=False))

print("\n### Failure Type Percentage Breakdown per Workload\n")
df_pof_valid = df_failed[df_failed['pof_type'].notna()]
if not df_pof_valid.empty:
    pof_workload = df_pof_valid.groupby(['workload', 'pof_type']).size().unstack(fill_value=0)
    pof_workload_pct = pof_workload.div(pof_workload.sum(axis=1), axis=0) * 100
    print(pof_workload_pct.round(2).to_markdown())
else:
    print("No valid failure classifications found.")

print("\n### Global Failure Type Percentage Breakdown\n")
if not df_pof_valid.empty:
    pof_global = df_pof_valid['pof_type'].value_counts(normalize=True) * 100
    df_pof_global = pof_global.reset_index()
    df_pof_global.columns = ['Failure Type', 'Percentage (%)']
    print(df_pof_global.to_markdown(index=False))
else:
    print("No valid global failure classifications found.")

### Task-by-Task Failure Classifications

| task_id            | sut_report   | pof_type                                               |
|:-------------------|:-------------|:-------------------------------------------------------|
| archeology-easy-11 | complete_R1  | Incorrect PIPELINE implementation/code generation      |
| archeology-easy-3  | without_R3   |                                                        |
| archeology-easy-6  | without_R1   | Lack of data CONTENT understanding                     |
| archeology-easy-6  | without_R2   |                                                        |
| archeology-easy-8  | complete_R1  | Incorrect PIPELINE implementation/code generation      |
| archeology-easy-8  | complete_R2  | Lack of data CONTENT understanding                     |
| archeology-easy-8  | complete_R3  | Lack of data CONTENT understanding                     |
| archeology-easy-8  | without_R1   | Incorrect PIPELINE implementation/code generation      |
| archeo

In [10]:
# Phase breakdown analysis
explore_total = df_all['explore_chars'].sum()
analyze_total = df_all['analyze_chars'].sum()
total_chars = explore_total + analyze_total

print("### Global Phase Character Breakdowns\n")
print(f"- **Exploration Phase:** {explore_total:,} chars ({explore_total/total_chars*100:.2f}%)")
print(f"- **Analysis Phase:** {analyze_total:,} chars ({analyze_total/total_chars*100:.2f}%)")

print("\n### Phase Breakdown by SUT Type\n")
sut_phase = df_all.groupby(df_all['sut_report'].str.startswith('complete').map({True: 'complete', False: 'without'}))[['explore_chars', 'analyze_chars']].sum()
sut_phase_pct = sut_phase.div(sut_phase.sum(axis=1), axis=0) * 100
print(sut_phase_pct.round(2).to_markdown())

### Global Phase Character Breakdowns

- **Exploration Phase:** 758,757 chars (26.72%)
- **Analysis Phase:** 2,081,030 chars (73.28%)

### Phase Breakdown by SUT Type

| sut_report   |   explore_chars |   analyze_chars |
|:-------------|----------------:|----------------:|
| complete     |           25.66 |           74.34 |
| without      |           27.82 |           72.18 |


In [11]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")

# Create figure for Accuracy Comparison
plt.figure(figsize=(10, 6))
df_acc = df_all.groupby(['workload', 'sut_report'])['success'].mean().reset_index()
df_acc['sut_type'] = df_acc['sut_report'].apply(lambda x: 'complete' if x.startswith('complete') else 'without')

sns.barplot(data=df_acc, x='workload', y='success', hue='sut_type', errorbar='sd', palette='muted')
plt.title('Accuracy Comparison across Workloads (Complete vs. Without)')
plt.ylabel('Mean Accuracy')
plt.xlabel('Workload')
plt.ylim(0, 1.0)
plt.tight_layout()
plt.savefig('repro_fle_2/accuracy_comparison.png', dpi=150)
plt.close()

# Create figure for Token Consumption per Workload
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_all, x='workload', y='tokens', hue='sut_label', palette='Set3')
plt.yscale('log')
plt.title('Token Consumption Distribution per Workload')
plt.ylabel('Tokens (log scale)')
plt.xlabel('Workload')
plt.tight_layout()
plt.savefig('repro_fle_2/token_consumption_distribution.png', dpi=150)
plt.close()

# Create figure for Point of Failure Distribution
plt.figure(figsize=(8, 8))
df_pof = df_all[(df_all['success'] == 0) & (df_all['pof_type'].notna())]
if not df_pof.empty:
    pof_counts = df_pof['pof_type'].value_counts()
    plt.pie(pof_counts, labels=pof_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
    plt.title('Global Point of Failure Type Distribution')
    plt.tight_layout()
    plt.savefig('repro_fle_2/point_of_failure_distribution.png', dpi=150)
plt.close()

print("Generated and saved charts: accuracy_comparison.png, token_consumption_distribution.png, point_of_failure_distribution.png")

Generated and saved charts: accuracy_comparison.png, token_consumption_distribution.png, point_of_failure_distribution.png
